Legal Contract Analysis Agent

Goal:

Transform retrieved legal contracts into actionable legal intelligence.

Capabilities:

1. Clause Extraction
2. Risk Detection
3. Missing Clause Detection
4. Contract Comparison
5. Executive Summary Generation

This notebook builds on Notebook 07.

In [1]:
!pip install -q \
langchain==0.3.26 \
langchain-core==0.3.68 \
langchain-community==0.3.27 \
langchain-openai==0.3.27 \
langchain-huggingface==0.1.2 \
langgraph==0.5.3 \
faiss-cpu \
pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 441.4/441.4 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.4/70.4 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 52.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.3/50.3 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.3 MB/s eta 0:00:00


In [2]:
import os
import pickle
import pandas as pd

from pathlib import Path
from typing import TypedDict
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.vectorstores import FAISS
from langgraph.graph import StateGraph
from langgraph.graph import END
from langchain_huggingface import HuggingFaceEmbeddings

Setting up Project Directory

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
PROJECT_DIR = Path("/content/drive/MyDrive/legal_ai")
VECTOR_DIR = PROJECT_DIR / "vectorstores"
METADATA_DIR = PROJECT_DIR / "metadata"
FAISS_DIR = VECTOR_DIR / "legal_faiss"

In [5]:
print("Project:", PROJECT_DIR)
print("Vector Store:", VECTOR_DIR)
print("Metadata:", METADATA_DIR)
print("FAISS:", FAISS_DIR)

Project: /content/drive/MyDrive/legal_ai
Vector Store: /content/drive/MyDrive/legal_ai/vectorstores
Metadata: /content/drive/MyDrive/legal_ai/metadata
FAISS: /content/drive/MyDrive/legal_ai/vectorstores/legal_faiss


Loading existing database


In [6]:
# Loading the cleaned chunks
with open("/content/drive/MyDrive/legal_ai/vectorstores/chunks.pkl","rb") as f:

    chunks = pickle.load(f)

print("Chunks:",len(chunks))

Chunks: 37818


In [7]:
#  loading the metadata
metadata_df = pd.read_csv("/content/drive/MyDrive/legal_ai/metadata/contract_type_mapping.csv")

print(metadata_df.shape)

(510, 2)


In [8]:
#  Embeddings
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [9]:
import os
os.environ["OPENAI_API_KEY"] ="bedrock-api-key-YmVkcm9jay5hbWF6b25hd3MuY29tLz9BY3Rpb249Q2FsbFdpdGhCZWFyZXJUb2tlbiZYLUFtei1BbGdvcml0aG09QVdTNC1ITUFDLVNIQTI1NiZYLUFtei1DcmVkZW50aWFsPUFTSUFYNjZWV1o0S1ZXNFlUUUJTJTJGMjAyNjA2MjMlMkZldS1ub3J0aC0xJTJGYmVkcm9jayUyRmF3czRfcmVxdWVzdCZYLUFtei1EYXRlPTIwMjYwNjIzVDA4NDc1OFomWC1BbXotRXhwaXJlcz00MzIwMCZYLUFtei1TZWN1cml0eS1Ub2tlbj1JUW9KYjNKcFoybHVYMlZqRUZFYUNtVjFMVzV2Y25Sb0xURWlSekJGQWlBeEl6RWtCY0E2JTJGdzRyckFKUzE3bXFBQ3FKb2RTeTBIbDNkNVNXNTF4MFVnSWhBTVZSblJXWUlkQUJYb1ZMZiUyRnRxellWMjllTEpSM09NVTNnJTJGMmFCanFrRWdLcWtEQ0JvUUFCb01OVFEzTlRFNU5qUTNOVEE1SWd3ZW94RDc1SEpGNmRuWG9Td3FoZ05RRFVtZVY3WGZqeU4lMkZ1Qjl1bHNUZ1oxVWtiZXpRJTJGOWEwVGlWYXZsdmZhZTRkcHNXekV0OGN6RHI3ViUyRkhFMUdHeFlYRlhJYXJGZEFGOVBPNWFQTHB4UTRrS1lkaTg5emFmNHR4JTJGVU5VeFgxSlBCbmVpTDZPQTc5alE1c1JCZ2JmRDhHWDc0Z05oTExqazUlMkY3bWk5MUtNajVESDJMJTJGZUh2eSUyRnlvZEMwMUpUJTJCMldaS2h4YzI2dVJxd0ViSm5xMUpvQ1VTc3phdGJ0UnNUNHhnNXFYMmx1dWZ5STR1d1IwWUxMTkRSUVhnUlg0c1JMUVdoOGY4OExUdHY5M2pjSVRmSldMU0lyUTRFVHhoNXBiM3dpbU50SUZGem5SYVh6MEdzQndrZjNyM2N0SnFLcFdoNHZ3QlM5aXU4akM3QnpFJTJGTUczYU5iJTJGRm0zTHZMcjVNMUgxZzdoMjlxdXE1a2U2TjI0MWxrZ0RWalRwZmFOMHdWcXFTZ2JGeCUyRlFXR09tU1JRdkpMYUQyRlVLazA2VkNnRWx3dHMlMkJMY2s4bnZIYmhQQjY4TThLSTVGZjBqWkFweUJlMnVoUVBMMjIzYThPMTV0JTJGMDBoJTJGUnV2aFpVdGltbHgzWjJkSEVIWU9IMFklMkJ4ZU1GVTNVS2JwRkFMc1hrWWN5Y01CSWIlMkJNQThCejZGVFo3VkElMkZ4dWxTb3dwNURwMFFZNjNnTEpyVGtDUEljWHFlME16cjhiaXlWdWpZOTB4T2NKMUtYM1VtbFZjS0hMbTlUTVVVMCUyRmQ0Rlk5WUhyakdZUlRyTXJvSXFFVkt5QjR3eWcyR3FWQmRzdTF6cFhGY09hZCUyRjQ2dnJTTDRVZFhKYyUyRjV1UiUyQlplRGlxcDQyUk5QSSUyQnJoMnozMW1ydzBwb0lpTUJ4RTVCQVJBZWxGdjJvJTJCMWlDczJybzdHekhuOTVFVk9QcW9oNG9xM3VaaHRLeHpmdjN6cW5sVzJLaElzdmR1NnFDbzRBalFvQzIlMkZubG9xSVV5c2tzZ1pPclpJeHAwVDFGZmFYa0RmZzYlMkYzYWpiZVhWZjBjVDdIcFhBSnhSb2hjY3E0cDE0UDg0dEtGdWZBdFF4TUxBeHY0ZHRuVDRtZXZxWnpIRHhQVlkyTGVtN01wT0x3a2JGRmQ4Q24wQnkxTk9Dd25TWWdib3dHeU9oMzZzZEtNalhZYnpBa2VSOGdIQXZDcDFrSkFJQzdCN3pYVGRsY2tIZndXaXNzT0tETUVNZEFDWk16ZHUzR2xBR01yZnVzZHN0TDVnZVVYNUZValRqOUVEUjFVYktkWHY0R25CNGg0R1ZydU5XWXJzQXNxU0pWRG40dyUzRCUzRCZYLUFtei1TaWduYXR1cmU9ODkyYzE3OTdhZGU2YWRjZThiYTlmY2VjZTI2NmVkMTBlNjc5ZTE0ZDgyZGVmMTcwZjgwNzkxYjRkZTliMDNlNyZYLUFtei1TaWduZWRIZWFkZXJzPWhvc3QmVmVyc2lvbj0x"
os.environ["OPENAI_BASE_URL"]= "https://bedrock-mantle.eu-north-1.api.aws/v1"


In [10]:
llm = ChatOpenAI(
    model="openai.gpt-oss-120b",
    temperature=0
)

Clause Extraction Agent

In [11]:
clause_extraction_prompt = ChatPromptTemplate.from_template(
"""
You are a senior legal contract reviewer.

Analyze the provided contract carefully.

For EACH clause below determine whether it is:

- Present
- Referenced
- Partially Present
- Missing

Definitions:

Present:
The clause appears directly in the provided text.

Referenced:
The clause does not appear directly,
but the document clearly incorporates
or references another agreement where
the clause likely exists.

Partially Present:
Only part of the clause appears,
or the clause is incomplete.

Missing:
The clause is neither present
nor referenced.

Review the following clauses:

1. Confidentiality
2. Termination
3. Force Majeure
4. Indemnification
5. Governing Law
6. Intellectual Property
7. Assignment
8. Dispute Resolution

For each clause provide:

STATUS:
Present / Referenced /
Partially Present / Missing

SUMMARY:
Brief explanation.

KEY OBLIGATIONS:
Main obligations created.

LEGAL RISK:
Low / Medium / High

WHY:
Explain why you selected that status.

Contract:
{contract}

Return the result as a professional markdown table.
"""
)

In [12]:
clause_extraction_chain =  clause_extraction_prompt | llm | StrOutputParser()

In [13]:
def extract_clauses(contract_text):
    """
    Extract major legal clauses from a contract text.
    """
    return clause_extraction_chain.invoke(
        {
            "contract": contract_text
        }
    )

In [14]:
contract_text = "\n\n".join(
    [
        chunk.page_content
        for chunk in chunks[:5]
    ]
)

result = extract_clauses(contract_text)
print(result)

**Contract Clause Review – Amendment #3 to the Manufacturing Agreement**  

| # | Clause | STATUS | SUMMARY | KEY OBLIGATIONS (as appear in this amendment or as likely carried over from the underlying Agreement) | LEGAL RISK | WHY |
|---|--------|--------|---------|------------------------------------------------------------|------------|-----|
| 1 | Confidentiality | **Present** (partial) | The amendment contains a “confidential treatment” statement that portions of the agreement are redacted and filed with the SEC under Rule 24b‑2. | • Parties must keep the redacted portions confidential and use them only as permitted by law. <br>• No further confidentiality obligations (e.g., for future disclosures) are set out in this amendment. | **Low‑Medium** – The clause exists but is limited to filing confidentiality; broader confidentiality duties are not restated, leaving reliance on the original Agreement. | The language explicitly addresses confidentiality of the redacted sections, so the 

In [15]:
risk_detection_prompt = ChatPromptTemplate.from_template(
"""
You are a senior legal risk analyst.

Review the contract text and identify legal and commercial risks.

Focus on:
- Missing confidentiality clause
- Missing termination rights
- Missing force majeure protection
- Missing indemnification
- Missing governing law
- Missing assignment restrictions
- Missing dispute resolution mechanism
- One-sided obligations
- Unlimited liability
- Weak protection of confidential information
- Weak IP ownership language

For each risk, provide:
1. Risk category
2. Why it is risky
3. Severity: Low / Medium / High
4. Practical recommendation

Contract text:
{contract}

Return structured markdown.
"""
)

In [16]:
risk_detection_chain = risk_detection_prompt | llm | StrOutputParser()

In [17]:
def detect_risks(contract_text):
    """
    Detect legal and commercial risks in a contract.
    """
    return risk_detection_chain.invoke(
        {
            "contract": contract_text
        }
    )

In [18]:
risk_result = detect_risks(contract_text)
print(risk_result)

## Legal & Commercial Risk Review – Amendment #3 to the Manufacturing Agreement  

| # | Risk Category | Why it is Risky | Severity* | Practical Recommendation |
|---|---------------|----------------|-----------|---------------------------|
| 1 | **Missing Confidentiality Clause** | The amendment only references “confidential treatment” for redacted portions but does **not** contain a stand‑alone confidentiality provision that obligates the parties to keep the rest of the amendment (e.g., supply volumes, pricing, liquidated‑damage amounts) confidential. Without a contractual confidentiality obligation, the parties may be forced to disclose material terms in litigation, to regulators, or to competitors, eroding competitive advantage. | **High** | Insert a comprehensive confidentiality clause (or incorporate the confidentiality provisions of the original Agreement by reference) that: <br>• Defines “Confidential Information” (including the amendment’s schedules, pricing, volumes, and liqu

In [19]:
missing_clause_prompt = ChatPromptTemplate.from_template(
"""
You are a senior legal contract reviewer.

Review the contract.

For each clause below determine:

- Present
- Referenced
- Partially Present
- Missing

Clauses:

1. Confidentiality
2. Termination
3. Force Majeure
4. Indemnification
5. Governing Law
6. Intellectual Property
7. Assignment
8. Dispute Resolution

Definitions:

Present:
Clause directly appears.

Referenced:
Clause likely exists in an incorporated
agreement or referenced document.

Partially Present:
Incomplete version appears.

Missing:
No evidence of clause.

For every clause provide:

STATUS

WHY IT MATTERS

POTENTIAL RISK

Contract:
{contract}

Return structured markdown.
"""
)

In [20]:
missing_clause_chain = missing_clause_prompt | llm | StrOutputParser()

In [21]:
def detect_missing_clauses(contract_text):
    """
    Detect which standard clauses are missing from a contract.
    """
    return missing_clause_chain.invoke(
        {
            "contract": contract_text
        }
    )

In [22]:
missing_result = detect_missing_clauses(contract_text)
print(missing_result)

## Contract‑Clause Review – Amendment #3 to the Manufacturing Agreement  
*(excerpt provided; full underlying Manufacturing Agreement not reproduced)*  

| # | Clause | Status* | Why it matters | Potential risk if not adequately addressed |
|---|--------|---------|----------------|-------------------------------------------|
| 1 | **Confidentiality** | **Partially Present** | The amendment contains a **confidential‑treatment notice** (“portions of this agreement … have been deleted and filed separately …”) and a statement that the parties agree to keep those portions confidential. However, there is **no substantive confidentiality clause** that defines the parties’ obligations, duration, permitted disclosures, or remedies for breach. | • Ambiguity about what information is protected and for how long.<br>• May leave the parties vulnerable to inadvertent disclosure or to disputes over whether a particular piece of information falls within the “confidential” carve‑out.<br>• If the underly

In [23]:
summary_prompt = ChatPromptTemplate.from_template(
"""
You are a legal executive advisor.

Write a concise but useful summary of the contract.

Include:
1. Purpose of the contract
2. Key obligations
3. Important risks
4. Important missing clauses
5. Practical recommendations

Write in clear business language.

Contract text:
{contract}

Return structured markdown.
"""
)

In [24]:
summary_chain = summary_prompt | llm | StrOutputParser()

In [25]:
def generate_summary(contract_text):
    """
    Generate a business-friendly legal summary.
    """
    return summary_chain.invoke(
        {
            "contract": contract_text
        }
    )

In [26]:
summary_result = generate_summary(contract_text)
print(summary_result)

## Summary of Amendment #3 to the Manufacturing Agreement  
*(ADMA BioManufacturing, LLC – “ADMA” and Sanofi Pasteur S.A. – “Sanofi Pasteur”)*  

| Item | Description |
|------|-------------|
| **Effective date** | 21 December 2017 |
| **Underlying agreements** | • Manufacturing Agreement (originally 30 Sep 2011, amended by Amendment #2 on 1 Aug 2016) – governs production of Rabies Fraction II Paste (the “Product”).<br>• Plasma Supply Agreement (20 Jan 2009, as amended) – governs supply of human Rabies Hyper‑immune Plasma to be used in the Product. |
| **Amendment focus** | Replace the “Prior Supply Terms” (Section 1 of Amendment #2) with a new, fixed‑quantity supply schedule and impose a minimum‑volume commitment for ADMA, together with a liquidated‑damages provision for shortfalls. |

---

### 1. Purpose of the Contract (Amendment #3)

To **re‑define the supply obligations** between ADMA and Sanofi Pasteur for the period **Q3 2018 – Q3 2019 (and a minimum‑volume requirement for Q4 20

In [27]:
comparison_prompt = ChatPromptTemplate.from_template(
"""
You are a senior legal contracts expert.

Compare Contract A and Contract B.

Focus on:

1. Contract Purpose
2. Confidentiality
3. Termination
4. Force Majeure
5. Indemnification
6. Governing Law
7. Intellectual Property
8. Assignment
9. Dispute Resolution
10. Liability Allocation

For each category explain:

- Present in A?
- Present in B?
- Major differences
- Which contract provides stronger protection
- Key legal risks

Then provide:

EXECUTIVE COMPARISON

KEY DIFFERENCES

MAJOR RISKS

RECOMMENDATION

Contract A:
{contract_a}

Contract B:
{contract_b}

Return professional markdown.
"""
)

In [28]:
comparison_chain = comparison_prompt | llm | StrOutputParser()

Helper Function

In [29]:
def get_contract_text(source_name, max_chunks=20):

    contract_chunks = []

    for chunk in chunks:

        if chunk.metadata["source"] == source_name:

            contract_chunks.append(
                chunk.page_content
            )

    return "\n\n".join(
        contract_chunks[:max_chunks]
    )

In [30]:
def compare_contracts(contract_a, contract_b):

    return comparison_chain.invoke(
        {
            "contract_a": contract_a,
            "contract_b": contract_b
        }
    )

In [31]:
contract_a = chunks[0].page_content
contract_b = chunks[500].page_content

comparison_result = compare_contracts(contract_a, contract_b)
print(comparison_result)

## CONTRACT COMPARISON – QUICK‑REFERENCE GUIDE  

| **Category** | **Present in Contract A?** | **Present in Contract B?** | **Major Differences** | **Stronger Protection** | **Key Legal Risks** |
|--------------|----------------------------|----------------------------|-----------------------|--------------------------|----------------------|
| **1. Contract Purpose** | Yes – an *Amendment #3* to a **Manufacturing Agreement** between ADMA BioManufacturing (successor to BPC) and Sanofi Pasteur. Purpose: modify/extend manufacturing services, pricing, timelines, etc. | No explicit purpose clause in the excerpt; the document appears to be a **Global Maintenance Master Agreement** (Aviation/Avionics) covering maintenance of aircraft systems. | Different industries & scope: pharma‑manufacturing vs. aviation‑maintenance. The purpose clause in A is clearly defined; B’s purpose is only implied by the title. | **A** – the purpose is expressly set out, limiting interpretation disputes. | • In A,

In [32]:
unique_sources = []

for chunk in chunks:

    source = chunk.metadata["source"]

    if source not in unique_sources:

        unique_sources.append(source)

for i, source in enumerate(unique_sources[:20]):

    print(i, source)

0 ADMA BioManufacturing, LLC -  Amendment #3 to Manufacturing Agreement .txt
1 ALLIANCEBANCORPINCOFPENNSYLVANIA_10_18_2006-EX-1.2-AGENCY AGREEMENT.txt
2 BELLRINGBRANDS,INC_02_07_2020-EX-10.18-MASTER SUPPLY AGREEMENT.txt
3 AlliedEsportsEntertainmentInc_20190815_8-K_EX-10.19_11788293_EX-10.19_Content License Agreement.txt
4 ArconicRolledProductsCorp_20191217_10-12B_EX-2.7_11923804_EX-2.7_Trademark License Agreement.txt
5 AzulSa_20170303_F-1A_EX-10.3_9943903_EX-10.3_Maintenance Agreement1.txt
6 ATMOSENERGYCORP_11_22_2002-EX-10.17-TRANSPORTATION SERVICE AGREEMENT.txt
7 AFSALABANCORPINC_08_01_1996-EX-1.1-AGENCY AGREEMENT.txt
8 ArcaUsTreasuryFund_20200207_N-2_EX-99.K5_11971930_EX-99.K5_Development Agreement.txt
9 AlliedEsportsEntertainmentInc_20190815_8-K_EX-10.34_11788308_EX-10.34_Sponsorship Agreement.txt
10 AimmuneTherapeuticsInc_20200205_8-K_EX-10.3_11967170_EX-10.3_Development Agreement.txt
11 AgapeAtpCorp_20191202_10-KA_EX-10.1_11911128_EX-10.1_Supply Agreement.txt
12 ADUROBIOTECH,INC_

In [33]:
source_a = unique_sources[0]
source_b = unique_sources[1]

print(source_a)
print(source_b)

ADMA BioManufacturing, LLC -  Amendment #3 to Manufacturing Agreement .txt
ALLIANCEBANCORPINCOFPENNSYLVANIA_10_18_2006-EX-1.2-AGENCY AGREEMENT.txt


In [34]:
contract_a = get_contract_text(
    source_a,
    max_chunks=20
)

contract_b = get_contract_text(
    source_b,
    max_chunks=20
)

In [35]:
comparison_result = compare_contracts(
    contract_a,
    contract_b
)

print("=" * 100)
print("CONTRACT A")
print(source_a)

print("\n")

print("CONTRACT B")
print(source_b)

print("=" * 100)

print(comparison_result)

CONTRACT A
ADMA BioManufacturing, LLC -  Amendment #3 to Manufacturing Agreement .txt


CONTRACT B
ALLIANCEBANCORPINCOFPENNSYLVANIA_10_18_2006-EX-1.2-AGENCY AGREEMENT.txt
## Comparative Analysis – Contract A vs. Contract B  

| **Category** | **Contract A** (Amendment #3 to the Manufacturing Agreement) | **Contract B** (Agency Agreement for Securities Offering) | **Major Differences** | **Which Provides Stronger Protection?** | **Key Legal Risks** |
|--------------|--------------------------------------------------------------|----------------------------------------------------------|-----------------------|------------------------------------------|----------------------|
| **1. Contract Purpose** | – Amends an existing **manufacturing & plasma‑supply** arrangement between AD BioManufacturing (successor to BPC) and Sanofi Pasteur. <br>– Sets a **minimum production volume**, updated supply plan, and a **compensation fee** for reduced volumes. | – Governs the **sale and distribution** 

In [36]:
def analyze_contract(contract_text):
    """
    Run full legal analysis on a single contract.
    """
    clauses = extract_clauses(contract_text)
    risks = detect_risks(contract_text)
    missing = detect_missing_clauses(contract_text)
    summary = generate_summary(contract_text)

    return {
        "clauses": clauses,
        "risks": risks,
        "missing_clauses": missing,
        "summary": summary
    }

In [37]:
analysis_result = analyze_contract(contract_text)

print("=" * 100)
print("CLAUSE EXTRACTION")
print("=" * 100)
print(analysis_result["clauses"])

print("\n" + "=" * 100)
print("RISK DETECTION")
print("=" * 100)
print(analysis_result["risks"])

print("\n" + "=" * 100)
print("MISSING CLAUSES")
print("=" * 100)
print(analysis_result["missing_clauses"])

print("\n" + "=" * 100)
print("SUMMARY")
print("=" * 100)
print(analysis_result["summary"])

CLAUSE EXTRACTION
**Contract Clause Review – Amendment #3 to the Manufacturing Agreement**  

| # | Clause | STATUS | SUMMARY | KEY OBLIGATIONS (if any) | LEGAL RISK | WHY |
|---|--------|--------|---------|--------------------------|------------|-----|
| 1 | Confidentiality | **Present** | The amendment contains a confidentiality notice stating that portions of the agreement are treated as confidential, have been redacted (“[***]”), and filed with the SEC under Rule 24b‑2. | • Parties must keep the redacted portions confidential.<br>• The redacted information is deemed “confidential treatment” and is subject to SEC filing requirements. | **Medium** | The clause is explicitly included, but it is limited to a filing‑related confidentiality notice and does not contain a full confidentiality‑obligation framework (e.g., duration, permitted disclosures). |
| 2 | Termination | **Missing** | No termination provision appears in the amendment, nor is there any reference to a termination clause 

In [38]:
source_name = chunks[0].metadata["source"]

contract_chunks = []

for chunk in chunks:
    if chunk.metadata["source"] == source_name:
        contract_chunks.append(chunk.page_content)

contract_text_full = "\n\n".join(contract_chunks[:10])

print("Source:", source_name)
print("Number of chunks collected:", len(contract_chunks))

analysis_result = analyze_contract(contract_text_full)

print("=" * 100)
print("CLAUSE EXTRACTION")
print("=" * 100)
print(analysis_result["clauses"])

print("\n" + "=" * 100)
print("RISK DETECTION")
print("=" * 100)
print(analysis_result["risks"])

print("\n" + "=" * 100)
print("MISSING CLAUSES")
print("=" * 100)
print(analysis_result["missing_clauses"])

print("\n" + "=" * 100)
print("SUMMARY")
print("=" * 100)
print(analysis_result["summary"])

Source: ADMA BioManufacturing, LLC -  Amendment #3 to Manufacturing Agreement .txt
Number of chunks collected: 20
CLAUSE EXTRACTION
**Contract Clause Review – Amendment #3 to the Manufacturing Agreement**  

| # | Clause | STATUS | SUMMARY | KEY OBLIGATIONS | LEGAL RISK | WHY |
|---|--------|--------|---------|-----------------|------------|-----|
| 1 | **Confidentiality** | **Partially Present** | The amendment contains a brief statement that “confidential treatment has been requested” for redacted portions and that those portions were filed with the SEC. No substantive confidentiality obligation (e.g., duty to keep information secret, permitted disclosures, duration) is set out. | • Parties acknowledge that certain sections are confidential and have been filed under Rule 24b‑2. <br>• No ongoing duty to protect other information is created in this amendment. | **Medium** | The amendment references confidentiality only in the context of SEC filing; the core confidentiality clause that 

In [39]:
source_a = chunks[0].metadata["source"]
source_b = chunks[500].metadata["source"]

contract_a_chunks = []
contract_b_chunks = []

for chunk in chunks:
    if chunk.metadata["source"] == source_a:
        contract_a_chunks.append(chunk.page_content)
    if chunk.metadata["source"] == source_b:
        contract_b_chunks.append(chunk.page_content)

contract_a_text = "\n\n".join(contract_a_chunks[:10])
contract_b_text = "\n\n".join(contract_b_chunks[:10])

comparison_result = compare_contracts(contract_a_text, contract_b_text)

print("=" * 100)
print("CONTRACT A:", source_a)
print("CONTRACT B:", source_b)
print("=" * 100)
print(comparison_result)

CONTRACT A: ADMA BioManufacturing, LLC -  Amendment #3 to Manufacturing Agreement .txt
CONTRACT B: AzulSa_20170303_F-1A_EX-10.3_9943903_EX-10.3_Maintenance Agreement1.txt
## QUICK‑LOOK COMPARISON  

| # | Issue | **Contract A** (Amendment #3 to the Manufacturing Agreement) | **Contract B** (Global Maintenance Master Agreement) | **Key Take‑aways** |
|---|-------|------------------------------------------------------------|------------------------------------------------------|--------------------|
| 1 | **Purpose** | amendment that revises the supply‑plan for a biologic product (Rabies Fraction II) and sets minimum‑volume commitments, liquidated‑damage penalties and a one‑time compensation fee. | master‑services/lease agreement for the maintenance, repair and leasing of ATR aircraft, LRUs, spare parts and related services. | A is a **product‑supply & manufacturing** contract; B is a **maintenance‑and‑lease** services contract. |
| 2 | **Confidentiality** | “Confidential treatment” for 

In [40]:
for i in range(10):

    print(
        i,
        chunks[i].metadata["source"]
    )

0 ADMA BioManufacturing, LLC -  Amendment #3 to Manufacturing Agreement .txt
1 ADMA BioManufacturing, LLC -  Amendment #3 to Manufacturing Agreement .txt
2 ADMA BioManufacturing, LLC -  Amendment #3 to Manufacturing Agreement .txt
3 ADMA BioManufacturing, LLC -  Amendment #3 to Manufacturing Agreement .txt
4 ADMA BioManufacturing, LLC -  Amendment #3 to Manufacturing Agreement .txt
5 ADMA BioManufacturing, LLC -  Amendment #3 to Manufacturing Agreement .txt
6 ADMA BioManufacturing, LLC -  Amendment #3 to Manufacturing Agreement .txt
7 ADMA BioManufacturing, LLC -  Amendment #3 to Manufacturing Agreement .txt
8 ADMA BioManufacturing, LLC -  Amendment #3 to Manufacturing Agreement .txt
9 ADMA BioManufacturing, LLC -  Amendment #3 to Manufacturing Agreement .txt


In [41]:
metadata_df.iloc[0]

,0
source,"ADMA BioManufacturing, LLC - Amendment #3 to ..."
contract_type,Manufacturing Agreement


In [42]:
source_name = metadata_df.iloc[0]["source"]

contract_text = get_contract_text(
    source_name
)

result = analyze_contract(
    contract_text
)

print(result["summary"])

**Amendment #3 – Manufacturing Agreement (ADMA BioManufacturing LLC & Sanofi Pasteur)**  
*Effective 21 Dec 2017*  

---  

## 1. Purpose of the Amendment  
To replace the “Prior Supply Terms” (Section 1 of Amendment #2) with a new, time‑bound production schedule for Rabies Fraction II Paste, to set a **minimum volume commitment**, to provide **liquidated‑damage** remedies for shortfalls, to compensate ADMA for the reduction in Sanofi’s original purchase commitment, and to realign risk‑allocation, liability caps, and the Quality Agreement in light of the updated supply plan.

---  

## 2. Key Obligations  

| Party | Obligation | Reference |
|-------|------------|-----------|
| **Sanofi Pasteur** | • Purchase a defined number of **[***] batches** of Product (see Exhibit A). <br>• Pay a **Compensation Fee** of **US $7 M** in five installments (see §5). | §2, §5 |
| **ADMA** | • Manufacture the agreed batches between **Q3 2018 – Q3 2019** (and a minimum volume for Q4 2018 – Q4 2019). <br

Legal Report Generation

In [43]:
legal_report_prompt = ChatPromptTemplate.from_template(
"""
You are a senior legal advisor.

Create a professional legal report.

Clause Analysis:
{clauses}

Risk Analysis:
{risks}

Missing Clauses:
{missing}

Summary:
{summary}

Produce:

1. Executive Summary

2. Major Clauses

3. Key Risks

4. Missing Clauses

5. Recommendations

Use professional legal language.
"""
)

In [44]:
legal_report_chain = legal_report_prompt | llm | StrOutputParser()


In [45]:
def generate_legal_report(
    clauses,
    risks,
    missing,
    summary
):

    return legal_report_chain.invoke(
        {
            "clauses": clauses,
            "risks": risks,
            "missing": missing,
            "summary": summary
        }
    )

In [46]:
def analyze_contract(contract_text):

    clauses = extract_clauses(contract_text)

    risks = detect_risks(contract_text)

    missing = detect_missing_clauses(contract_text)

    summary = generate_summary(contract_text)

    report = generate_legal_report(
        clauses,
        risks,
        missing,
        summary
    )

    return {
        "clauses": clauses,
        "risks": risks,
        "missing": missing,
        "summary": summary,
        "report": report
    }

In [47]:
analysis = analyze_contract(
    get_contract_text(
        metadata_df.iloc[0]["source"]
    )
)

print(
    analysis["report"]
)

**LEGAL REPORT**  
**Amendment #3 to the Manufacturing Agreement**  
**Parties:** ADMA BioManufacturing, LLC (“ADMA”) – a Delaware‑registered LLC; Sanofi Pasteur S.A. (“Sanofi Pasteur”) – a French corporation.  
**Effective Date:** 21 December 2017  

Prepared by: **Senior Legal Advisor**  
Date: 23 June 2026  

---  

## 1. Executive Summary  

Amendment #3 replaces the “Prior Supply Terms” of the underlying Manufacturing Agreement with a fixed‑volume, time‑bound production schedule for Rabies Fraction II Paste, introduces liquidated‑damage penalties for missed batches or schedule, provides a $7 million “Compensation Fee” to ADMA, reallocates risk of loss for source plasma, and inserts a unilateral termination right for Sanofi Pasteur upon escalation of ADMA’s FDA compliance status.  

While the amendment contains a clear termination trigger and references the existing indemnity provision, it **omits a number of core boiler‑plate clauses** (confidentiality, force‑majeure, governing la

In [48]:
def legal_contract_agent(contract_text):

    return analyze_contract(
        contract_text
    )

In [49]:
def legal_comparison_agent(
    contract_a,
    contract_b
):

    return compare_contracts(
        contract_a,
        contract_b
    )

In [50]:
def legal_contract_agent(task, contract_text=None, contract_b=None):

    task = task.strip().lower()

    if task in {"analysis", "full_analysis", "analyze"}:
        if contract_text is None:
            raise ValueError("contract_text is required for full analysis.")
        return analyze_contract(contract_text)

    if task == "clauses":
        if contract_text is None:
            raise ValueError("contract_text is required for clause extraction.")
        return extract_clauses(contract_text)

    if task == "risks":
        if contract_text is None:
            raise ValueError("contract_text is required for risk detection.")
        return detect_risks(contract_text)

    if task in {"missing", "missing_clauses"}:
        if contract_text is None:
            raise ValueError("contract_text is required for missing clause detection.")
        return detect_missing_clauses(contract_text)

    if task == "summary":
        if contract_text is None:
            raise ValueError("contract_text is required for summary generation.")
        return generate_summary(contract_text)

    if task in {"compare", "comparison"}:
        if contract_text is None or contract_b is None:
            raise ValueError("Both contract_text and contract_b are required for comparison.")
        return compare_contracts(contract_text, contract_b)

    raise ValueError(
        "Unsupported task. Use one of: analysis, clauses, risks, missing, summary, compare."
    )

In [51]:
source_name = metadata_df.iloc[0]["source"]
contract_text = get_contract_text(source_name, max_chunks=20)

analysis = legal_contract_agent("full_analysis", contract_text=contract_text)

print("=" * 100)
print("CLAUSES")
print("=" * 100)
print(analysis["clauses"])

print("\n" + "=" * 100)
print("RISKS")
print("=" * 100)
print(analysis["risks"])

print("\n" + "=" * 100)
print("MISSING CLAUSES")
print("=" * 100)
print(analysis["missing"])

print("\n" + "=" * 100)
print("SUMMARY")
print("=" * 100)
print(analysis["summary"])

print("\n" + "=" * 100)
print("FINAL REPORT")
print("=" * 100)
print(analysis["report"])

CLAUSES
**Contract‑Clause Review – Amendment #3 to the Manufacturing Agreement**  

| # | Clause | Status | Summary (what the amendment says / does) | Key Obligations Created | Legal Risk* | Why |
|---|--------|--------|------------------------------------------|--------------------------|-------------|-----|
| 1 | Confidentiality | **Present** | The amendment states that portions marked “[***]” are treated as confidential and have been filed with the SEC under Rule 24b‑2. The parties therefore agree to keep those redacted sections confidential. | • Keep all redacted information confidential.<br>• Comply with SEC filing confidentiality requirements. | **Medium** | A confidentiality provision exists, but it is limited to the redacted excerpts and does not contain the usual broader confidentiality obligations (e.g., duration, permitted disclosures). |
| 2 | Termination | **Present** | A termination right is inserted: if ADMA’s FDA compliance status is escalated (or ADMA fails to supply t

In [52]:
source_a = metadata_df.iloc[0]["source"]
source_b = metadata_df.iloc[1]["source"]

contract_a = get_contract_text(source_a, max_chunks=20)
contract_b = get_contract_text(source_b, max_chunks=20)

comparison = legal_contract_agent(
    "compare",
    contract_text=contract_a,
    contract_b=contract_b
)

print("=" * 100)
print("CONTRACT A:", source_a)
print("CONTRACT B:", source_b)
print("=" * 100)
print(comparison)

CONTRACT A: ADMA BioManufacturing, LLC -  Amendment #3 to Manufacturing Agreement .txt
CONTRACT B: ALLIANCEBANCORPINCOFPENNSYLVANIA_10_18_2006-EX-1.2-AGENCY AGREEMENT.txt
## CONTRACT COMPARISON – QUICK‑REFERENCE TABLE  

| Category | Contract A (Manufacturing Amendment) | Contract B (Agency / Securities Offering) | Which offers stronger protection? | Key Legal Risks |
|----------|--------------------------------------|-------------------------------------------|-----------------------------------|-----------------|
| **1. Purpose** | Amendment to a 2011 Manufacturing Agreement – sets new supply volumes, pricing, liquidated‑damage provisions, and payment schedule for the production of Rabies Fraction II Paste. | Agency agreement that authorises Sandler O’Neill to sell up to 2.45 M (potentially 2.81 M) shares of the Company’s common stock in a “Reorganization and Additional Stock Issuance.” | **A** – more detailed, performance‑based obligations; B is limited to representation and marketi

 LEGAL CONTRACT ANALYSIS AGENT


In [53]:
def clause_agent(contract_text):

    return {
        "agent": "Clause Agent",
        "output": extract_clauses(contract_text)
    }

In [54]:
def risk_agent(contract_text):

    return {
        "agent": "Risk Agent",
        "output": detect_risks(contract_text)
    }

In [55]:
def missing_clause_agent(contract_text):

    return {
        "agent": "Missing Clause Agent",
        "output": detect_missing_clauses(contract_text)
    }

In [56]:
def summary_agent(contract_text):

    return {
        "agent": "Summary Agent",
        "output": generate_summary(contract_text)
    }

In [57]:
def report_agent(
    clauses,
    risks,
    missing,
    summary
):

    return {
        "agent": "Report Agent",
        "output": generate_legal_report(
            clauses,
            risks,
            missing,
            summary
        )
    }

In [59]:
# Comparison Agent
def comparison_agent(
    contract_a,
    contract_b
):

    return {
        "agent": "Comparison Agent",
        "output": compare_contracts(
            contract_a,
            contract_b
        )
    }

In [60]:
# Master Comparison Agent
def legal_comparison_agent(
    contract_a,
    contract_b
):

    print("=" * 80)
    print("LEGAL CONTRACT COMPARISON AGENT")
    print("=" * 80)

    result = comparison_agent(
        contract_a,
        contract_b
    )

    return {
        "comparison_report":
            result["output"]
    }

In [61]:
source_a = metadata_df.iloc[0]["source"]
source_b = metadata_df.iloc[1]["source"]

contract_a = get_contract_text(
    source_a,
    max_chunks=20
)

contract_b = get_contract_text(
    source_b,
    max_chunks=20
)

comparison_result = legal_comparison_agent(
    contract_a,
    contract_b
)

print(
    comparison_result[
        "comparison_report"
    ]
)

LEGAL CONTRACT COMPARISON AGENT
## Comparative Analysis – Contract A vs. Contract B  

| **Category** | **Contract A** (Amendment #3 to the Manufacturing Agreement) | **Contract B** (Agency Agreement for Securities Offering) | **Major Differences** | **Which Provides Stronger Protection?** | **Key Legal Risks** |
|--------------|--------------------------------------------------------------|----------------------------------------------------------|-----------------------|------------------------------------------|----------------------|
| **1. Contract Purpose** | – Amends an existing **manufacturing & plasma‑supply** arrangement between AD BioManufacturing (successor to BPC) and Sanofi Pasteur. <br>– Sets a **minimum production volume**, updated supply plan, and a **compensation fee** for reduced volumes. | – Creates an **agency relationship** between the Company (Alliance Bancorp) and Sandler O’Neill to **sell up to 2.45 M shares** of the Company’s common stock (with possible increa

In [62]:
# ==========================================================
# PRODUCTION READY
# ==========================================================

def analyze_contract(contract_text):
    """
    Run full legal analysis on a single contract.
    Returns both 'missing' and 'missing_clauses' for backward compatibility.
    """
    clauses = extract_clauses(contract_text)
    risks = detect_risks(contract_text)
    missing = detect_missing_clauses(contract_text)
    summary = generate_summary(contract_text)

    report = generate_legal_report(
        clauses,
        risks,
        missing,
        summary
    )

    return {
        "clauses": clauses,
        "risks": risks,
        "missing": missing,
        "missing_clauses": missing,  # backward compatibility for old cells
        "summary": summary,
        "report": report
    }


def legal_contract_agent(task_or_contract_text, contract_text=None, contract_b=None):
    """
    Backward-compatible dispatcher.

    Supports:
    1) legal_contract_agent("full_analysis", contract_text=...)
    2) legal_contract_agent("clauses", contract_text=...)
    3) legal_contract_agent("risks", contract_text=...)
    4) legal_contract_agent("missing", contract_text=...)
    5) legal_contract_agent("summary", contract_text=...)
    6) legal_contract_agent("compare", contract_text=..., contract_b=...)
    7) Old style: legal_contract_agent(contract_text) for full analysis
    """
    known_tasks = {"analysis", "full_analysis", "analyze", "clauses", "risks", "missing", "missing_clauses", "summary", "compare", "comparison"}

    # Backward compatibility for old single-argument calls
    if contract_text is None and contract_b is None and isinstance(task_or_contract_text, str):
        maybe_text = task_or_contract_text.strip()
        if len(maybe_text) > 80 and task_or_contract_text.lower().strip() not in known_tasks:
            return analyze_contract(task_or_contract_text)

    task = (task_or_contract_text or "").strip().lower()

    if task in {"analysis", "full_analysis", "analyze"}:
        if contract_text is None:
            raise ValueError("contract_text is required for full analysis.")
        return analyze_contract(contract_text)

    if task == "clauses":
        if contract_text is None:
            raise ValueError("contract_text is required for clause extraction.")
        return extract_clauses(contract_text)

    if task == "risks":
        if contract_text is None:
            raise ValueError("contract_text is required for risk detection.")
        return detect_risks(contract_text)

    if task in {"missing", "missing_clauses"}:
        if contract_text is None:
            raise ValueError("contract_text is required for missing clause detection.")
        return detect_missing_clauses(contract_text)

    if task == "summary":
        if contract_text is None:
            raise ValueError("contract_text is required for summary generation.")
        return generate_summary(contract_text)

    if task in {"compare", "comparison"}:
        if contract_text is None or contract_b is None:
            raise ValueError("Both contract_text and contract_b are required for comparison.")
        return compare_contracts(contract_text, contract_b)

    raise ValueError(
        "Unsupported task. Use one of: analysis, clauses, risks, missing, summary, compare."
    )


def legal_comparison_agent(contract_a, contract_b):
    """
    Clean comparison wrapper for Streamlit.
    Returns both a named key and a generic key for flexibility.
    """
    comparison_report = compare_contracts(contract_a, contract_b)
    return {
        "comparison_report": comparison_report,
        "report": comparison_report
    }